In [0]:
dbutils.widgets.text("catalog", "global_airline", "Catalog Name")
catalog = dbutils.widgets.get("catalog")

In [0]:
# silver.region
from pyspark.sql.functions import *
from pyspark.sql.window import Window

region_df = spark.read.table(f"{catalog}.bronze.region")
standardized_region_df = (
    region_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "r_regionkey": expr("NULLIF(TRIM(r_regionkey), '')").cast("int"),
        "r_name": trim(col("r_name")),
        "r_comment": trim(col("r_comment"))
    })
)

# Flag records for quarantine: r_regionkey is null or duplicate
w = Window.partitionBy("r_regionkey")
standardized_region_df = standardized_region_df.withColumn(
    "_key_count", count("*").over(w)
)

standardized_region_df = standardized_region_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("r_regionkey").isNull(), lit("r_regionkey is null")).otherwise(lit(None).cast("string")),
        when(col("_key_count") > 1, lit("duplicate r_regionkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_df = standardized_region_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count")

clean_df = standardized_region_df.filter(
    col("r_regionkey").isNotNull() & (col("_key_count") == 1)
).drop("_key_count", "_quarantine_reason")

# Write quarantined records
quarantine_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region_quarantine")

# Write clean records
clean_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.region")


In [0]:
# silver.nation
from pyspark.sql.functions import *
from pyspark.sql.window import Window

nation_df = spark.read.table(f"{catalog}.bronze.nation")
standardized_nation_df = (
    nation_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "n_nationkey": expr("NULLIF(TRIM(n_nationkey), '')").cast("int"),
        "n_name": trim(col("n_name")),
        "n_regionkey": expr("NULLIF(TRIM(n_regionkey), '')").cast("int"),
        "n_comment": trim(col("n_comment"))
    })
)

# Flag records for quarantine: n_nationkey is null or duplicate
w = Window.partitionBy("n_nationkey")
standardized_nation_df = standardized_nation_df.withColumn(
    "_key_count", count("*").over(w)
)

# Referential integrity: n_regionkey must match r_regionkey in silver.region
valid_region_df = spark.read.table(f"{catalog}.silver.region").select(
    col("r_regionkey").alias("_valid_regionkey")
)
standardized_nation_df = standardized_nation_df.join(
    valid_region_df,
    standardized_nation_df.n_regionkey == valid_region_df._valid_regionkey,
    "left"
).withColumn(
    "_invalid_regionkey",
    when(
        col("n_regionkey").isNotNull() & col("_valid_regionkey").isNull(),
        lit("n_regionkey does not match any r_regionkey in silver.region")
    ).otherwise(lit(None).cast("string"))
)

standardized_nation_df = standardized_nation_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("n_nationkey").isNull(), lit("n_nationkey is null")).otherwise(lit(None).cast("string")),
        when(col("_key_count") > 1, lit("duplicate n_nationkey")).otherwise(lit(None).cast("string")),
        when(col("_invalid_regionkey").isNotNull(), col("_invalid_regionkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_nation_df = standardized_nation_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_regionkey", "_invalid_regionkey")

clean_nation_df = standardized_nation_df.filter(
    col("n_nationkey").isNotNull() & (col("_key_count") == 1) & col("_valid_regionkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_regionkey", "_invalid_regionkey")

# Write quarantined records
quarantine_nation_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation_quarantine")

# Write clean records
clean_nation_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nation")


In [0]:
# silver.passenger
from pyspark.sql.functions import *
from pyspark.sql.window import Window

passenger_df = spark.read.table(f"{catalog}.bronze.passenger")
standardized_passenger_df = (
    passenger_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "pa_passengerid": expr("NULLIF(TRIM(pa_passengerid), '')").cast("int"),
        "pa_name": trim(col("pa_name")),
        "pa_address": trim(col("pa_address")),
        "pa_nationkey": expr("NULLIF(TRIM(pa_nationkey), '')").cast("int"),
        "pa_phone": trim(col("pa_phone")),
        "pa_loyaltypoints": trim(col("pa_loyaltypoints")).cast("decimal(10,2)"),
        "pa_travelpurpose": trim(col("pa_travelpurpose")),
        "pa_comment": trim(col("pa_comment"))
    })
)

# Flag records for quarantine: pa_passengerid is null or duplicate
w = Window.partitionBy("pa_passengerid").orderBy("pa_passengerid")
standardized_passenger_df = standardized_passenger_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: pa_nationkey must match n_nationkey in silver.nation
valid_nation_df = spark.read.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_valid_nationkey")
)
standardized_passenger_df = standardized_passenger_df.join(
    valid_nation_df,
    standardized_passenger_df.pa_nationkey == valid_nation_df._valid_nationkey,
    "left"
).withColumn(
    "_invalid_nationkey",
    when(
        col("pa_nationkey").isNotNull() & col("_valid_nationkey").isNull(),
        lit("pa_nationkey does not match any n_nationkey in silver.nation")
    ).otherwise(lit(None).cast("string"))
)

standardized_passenger_df = standardized_passenger_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("pa_passengerid").isNull(), lit("pa_passengerid is null")).otherwise(lit(None).cast("string")),
        when(col("pa_name").isNull(), lit("pa_name is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate pa_passengerid")).otherwise(lit(None).cast("string")),
        when(col("_invalid_nationkey").isNotNull(), col("_invalid_nationkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_passenger_df = standardized_passenger_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_nationkey", "_invalid_nationkey", "_row_num")

clean_passenger_df = standardized_passenger_df.filter(
    col("pa_passengerid").isNotNull() & col("pa_name").isNotNull() & (col("_row_num") == 1) & col("_valid_nationkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_nationkey", "_invalid_nationkey", "_row_num")

# Write quarantined records
quarantine_passenger_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.passenger_quarantine")

# Write clean records
clean_passenger_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.passenger")

In [0]:
# silver.route
from pyspark.sql.functions import *
from pyspark.sql.window import Window

route_df = spark.read.table(f"{catalog}.bronze.route")
standardized_route_df = (
    route_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "ro_routeid": expr("NULLIF(TRIM(ro_routeid), '')").cast("int"),
        "ro_name": trim(col("ro_name")),
        "ro_aircraft": trim(col("ro_aircraft")),
        "ro_airline_brand": trim(col("ro_airline_brand")),
        "ro_routetype": trim(col("ro_routetype")),
        "ro_capacity": trim(col("ro_capacity")).cast("int"),
        "ro_cabinconfig": trim(col("ro_cabinconfig")),
        "ro_baseprice": trim(col("ro_baseprice")).cast("decimal(10,2)"),
        "ro_comment": trim(col("ro_comment"))
    })
)

# Flag records for quarantine: ro_routeid is null or duplicate
w = Window.partitionBy("ro_routeid").orderBy("ro_routeid")
standardized_route_df = standardized_route_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

standardized_route_df = standardized_route_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("ro_routeid").isNull(), lit("ro_routeid is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate ro_routeid")).otherwise(lit(None).cast("string")),
        when(col("ro_baseprice") <= 0, lit("ro_baseprice is zero or negative")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_route_df = standardized_route_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_row_num")

clean_route_df = standardized_route_df.filter(
    col("ro_routeid").isNotNull() & (col("_row_num") == 1) & (col("ro_baseprice") > 0)
).drop("_key_count", "_quarantine_reason", "_row_num")

# Write quarantined records
quarantine_route_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.route_quarantine")

# Write clean records
clean_route_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.route")


In [0]:
# silver.airline
from pyspark.sql.functions import *
from pyspark.sql.window import Window

airline_df = spark.read.table(f"{catalog}.bronze.airline")
standardized_airline_df = (
    airline_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "al_airlineid": expr("NULLIF(TRIM(al_airlineid), '')").cast("int"),
        "al_name": trim(col("al_name")),
        "al_address": trim(col("al_address")),
        "al_nationkey": expr("NULLIF(TRIM(al_nationkey), '')").cast("int"),
        "al_phone": trim(col("al_phone")),
        "al_rating": trim(col("al_rating")).cast("decimal(10,2)"),
        "al_comment": trim(col("al_comment"))
    })
)

# Flag records for quarantine: al_airlineid is null or duplicate
w = Window.partitionBy("al_airlineid").orderBy("al_airlineid")
standardized_airline_df = standardized_airline_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: al_nationkey must match n_nationkey in silver.nation
valid_nation_df = spark.read.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_valid_nationkey")
)
standardized_airline_df = standardized_airline_df.join(
    valid_nation_df,
    standardized_airline_df.al_nationkey == valid_nation_df._valid_nationkey,
    "left"
).withColumn(
    "_invalid_nationkey",
    when(
        col("al_nationkey").isNotNull() & col("_valid_nationkey").isNull(),
        lit("al_nationkey does not match any n_nationkey in silver.nation")
    ).otherwise(lit(None).cast("string"))
)

standardized_airline_df = standardized_airline_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("al_airlineid").isNull(), lit("al_airlineid is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate al_airlineid")).otherwise(lit(None).cast("string")),
        when(col("_invalid_nationkey").isNotNull(), col("_invalid_nationkey")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_airline_df = standardized_airline_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_nationkey", "_invalid_nationkey", "_row_num")

clean_airline_df = standardized_airline_df.filter(
    col("al_airlineid").isNotNull() & (col("_row_num") == 1) & col("_valid_nationkey").isNotNull()
).drop("_key_count", "_quarantine_reason", "_valid_nationkey", "_invalid_nationkey", "_row_num")

# Write quarantined records
quarantine_airline_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airline_quarantine")

# Write clean records
clean_airline_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airline")

In [0]:
# silver.airline_route
from pyspark.sql.functions import *
from pyspark.sql.window import Window

airline_route_df = spark.read.table(f"{catalog}.bronze.airline_route")
standardized_airline_route_df = (
    airline_route_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id")
    .withColumns({
        "ar_airlineid": expr("NULLIF(TRIM(ar_airlineid), '')").cast("int"),
        "ar_routeid": expr("NULLIF(TRIM(ar_routeid), '')").cast("int"),
        "ar_available_seats": trim(col("ar_available_seats")).cast("int"),
        "ar_operatingcost": trim(col("ar_operatingcost")).cast("decimal(10,2)"),
        "ar_comment": trim(col("ar_comment"))
    })
)

# Flag records for quarantine: no primary key, so deduplicate on the entire row
w = Window.partitionBy("ar_airlineid", "ar_routeid").orderBy("ar_airlineid", "ar_routeid")
standardized_airline_route_df = standardized_airline_route_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: ar_airlineid must match al_airlineid in silver.airline
valid_airline_df = spark.read.table(f"{catalog}.silver.airline").select(
    col("al_airlineid").alias("_valid_airlineid")
)
# Referential integrity: ar_routeid must match ro_routeid in silver.route
valid_route_df = spark.read.table(f"{catalog}.silver.route").select(
    col("ro_routeid").alias("_valid_routeid")
)

standardized_airline_route_df = standardized_airline_route_df.join(
    valid_airline_df,
    standardized_airline_route_df.ar_airlineid == valid_airline_df._valid_airlineid,
    "left"
).join(
    valid_route_df,
    standardized_airline_route_df.ar_routeid == valid_route_df._valid_routeid,
    "left"
).withColumn(
    "_invalid_airlineid",
    when(
        col("ar_airlineid").isNotNull() & col("_valid_airlineid").isNull(),
        lit("ar_airlineid does not match any al_airlineid in silver.airline")
    ).otherwise(lit(None).cast("string"))
).withColumn(
    "_invalid_routeid",
    when(
        col("ar_routeid").isNotNull() & col("_valid_routeid").isNull(),
        lit("ar_routeid does not match any ro_routeid in silver.route")
    ).otherwise(lit(None).cast("string"))
)

standardized_airline_route_df = standardized_airline_route_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate row")).otherwise(lit(None).cast("string")),
        when(col("_invalid_airlineid").isNotNull(), col("_invalid_airlineid")).otherwise(lit(None).cast("string")),
        when(col("_invalid_routeid").isNotNull(), col("_invalid_routeid")).otherwise(lit(None).cast("string")),
        when(col("ar_operatingcost") <= 0, lit("ar_operatingcost is zero or negative")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_airline_route_df = standardized_airline_route_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_row_num", "_valid_airlineid", "_valid_routeid", "_invalid_airlineid", "_invalid_routeid")

clean_airline_route_df = standardized_airline_route_df.filter(
    (col("_row_num") == 1)
    & col("_valid_airlineid").isNotNull()
    & col("_valid_routeid").isNotNull()
    & (col("ar_operatingcost") > 0)
).drop("_key_count", "_quarantine_reason", "_row_num", "_valid_airlineid", "_valid_routeid", "_invalid_airlineid", "_invalid_routeid")

# Write quarantined records
quarantine_airline_route_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airline_route_quarantine")

# Write clean records
clean_airline_route_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airline_route")

In [0]:
# silver.airline
from pyspark.sql.functions import *
from pyspark.sql.window import Window

booking_df = spark.read.table(f"{catalog}.bronze.booking")
standardized_booking_df = (
    booking_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id", "_rescued_data")
    .withColumns({
        "bk_bookingid": expr("NULLIF(TRIM(bk_bookingid), '')").cast("int"),
        "bk_passengerid": expr("NULLIF(TRIM(bk_passengerid), '')").cast("int"),
        "bk_status": trim(col("bk_status")),
        "bk_totalprice": trim(col("bk_totalprice")).cast("decimal(10,2)"),
        "bk_bookingdate": try_to_date(col("bk_bookingdate"), "yyyy-MM-dd"),
        "bk_flightdate": try_to_date(col("bk_flightdate"), "yyyy-MM-dd"),
        "bk_bookingpriority": trim(col("bk_bookingpriority")),
        "bk_agent": trim(col("bk_agent")),
        "bk_comment": trim(col("bk_comment"))
    })
)

# Flag records for quarantine: bk_bookingid is null or duplicate
w = Window.partitionBy("bk_bookingid").orderBy("bk_bookingid")
standardized_booking_df = standardized_booking_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: bk_passengerid must match pa_passengerid in silver.passenger
valid_passenger_df = spark.read.table(f"{catalog}.silver.passenger").select(
    col("pa_passengerid").alias("_valid_passengerid")
)
standardized_booking_df = standardized_booking_df.join(
    valid_passenger_df,
    standardized_booking_df.bk_passengerid == valid_passenger_df._valid_passengerid,
    "left"
).withColumn(
    "_invalid_passengerid",
    when(
        col("bk_passengerid").isNotNull() & col("_valid_passengerid").isNull(),
        lit("bk_passengerid does not match any pa_passengerid in silver.passenger")
    ).otherwise(lit(None).cast("string"))
)

standardized_booking_df = standardized_booking_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("bk_bookingid").isNull(), lit("bk_bookingid is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate bk_bookingid")).otherwise(lit(None).cast("string")),
        when(col("bk_bookingdate") > col("bk_flightdate"), lit("bk_bookingdate is greater than bk_flightdate")).otherwise(lit(None).cast("string")),
        when(col("bk_totalprice") <= 0, lit("bk_totalprice is zero or negative")).otherwise(lit(None).cast("string")),
        when(col("_invalid_passengerid").isNotNull(), col("_invalid_passengerid")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_booking_df = standardized_booking_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_passengerid", "_invalid_passengerid", "_row_num")

clean_booking_df = standardized_booking_df.filter(
    col("bk_bookingid").isNotNull() & (col("_row_num") == 1) & col("_valid_passengerid").isNotNull()
    & (col("bk_bookingdate") <= col("bk_flightdate"))
    & (col("bk_totalprice") > 0)
).drop("_key_count", "_quarantine_reason", "_valid_passengerid", "_invalid_passengerid", "_row_num")

# Write quarantined records
quarantine_booking_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.booking_quarantine")

# Write clean records
clean_booking_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.booking")

In [0]:
# silver.flight_booking
from pyspark.sql.functions import *
from pyspark.sql.window import Window

flight_booking_df = spark.read.table(f"{catalog}.bronze.flight_booking")
standardized_flight_booking_df = (
    flight_booking_df
    .drop("_source_file", "_ingested_at", "_pipeline_run_id", "_rescued_data")
    .withColumns({
        "fb_bookingid": expr("NULLIF(TRIM(fb_bookingid), '')").cast("int"),
        "fb_routeid": expr("NULLIF(TRIM(fb_routeid), '')").cast("int"),
        "fb_airlineid": expr("NULLIF(TRIM(fb_airlineid), '')").cast("int"),
        "fb_linenumber": expr("NULLIF(TRIM(fb_linenumber), '')").cast("int"),
        "fb_seats": expr("NULLIF(TRIM(fb_seats), '')").cast("int"),
        "fb_ticketprice": trim(col("fb_ticketprice")).cast("decimal(10,2)"),
        "fb_discount": trim(col("fb_discount")).cast("decimal(10,2)"),
        "fb_tax": trim(col("fb_tax")).cast("decimal(10,2)"),
        "fb_refundflag": trim(col("fb_refundflag")),
        "fb_segmentstatus": trim(col("fb_segmentstatus")),
        "fb_departuredate": try_to_date(col("fb_departuredate"), "yyyy-MM-dd"),
        "fb_commitdate": try_to_date(col("fb_commitdate"), "yyyy-MM-dd"),
        "fb_arrivaldate": try_to_date(col("fb_arrivaldate"), "yyyy-MM-dd"),
        "fb_cabinclass": trim(col("fb_cabinclass")),
        "fb_comment": trim(col("fb_comment"))
    })
)

# Deduplicate on the entire row: keep one, rest to quarantine
w = Window.partitionBy(
    "fb_bookingid", "fb_routeid", "fb_airlineid", "fb_linenumber",
    "fb_seats", "fb_ticketprice", "fb_discount", "fb_tax",
    "fb_refundflag", "fb_segmentstatus", "fb_departuredate",
    "fb_commitdate", "fb_arrivaldate", "fb_cabinclass", "fb_comment"
).orderBy("fb_bookingid")
standardized_flight_booking_df = standardized_flight_booking_df.withColumn(
    "_key_count", count("*").over(w)
).withColumn(
    "_row_num", row_number().over(w)
)

# Referential integrity: fb_bookingid must match bk_bookingid in silver.booking
valid_booking_df = spark.read.table(f"{catalog}.silver.booking").select(
    col("bk_bookingid").alias("_valid_bookingid")
)
standardized_flight_booking_df = standardized_flight_booking_df.join(
    valid_booking_df,
    standardized_flight_booking_df.fb_bookingid == valid_booking_df._valid_bookingid,
    "left"
).withColumn(
    "_invalid_bookingid",
    when(
        col("fb_bookingid").isNotNull() & col("_valid_bookingid").isNull(),
        lit("fb_bookingid does not match any bk_bookingid in silver.booking")
    ).otherwise(lit(None).cast("string"))
)

standardized_flight_booking_df = standardized_flight_booking_df.withColumn(
    "_quarantine_reason",
    nullif(concat_ws(
        ", ",
        when(col("fb_bookingid").isNull(), lit("fb_bookingid is null")).otherwise(lit(None).cast("string")),
        when(col("fb_routeid").isNull(), lit("fb_routeid is null")).otherwise(lit(None).cast("string")),
        when(col("fb_airlineid").isNull(), lit("fb_airlineid is null")).otherwise(lit(None).cast("string")),
        when(col("fb_linenumber").isNull(), lit("fb_linenumber is null")).otherwise(lit(None).cast("string")),
        when((col("_key_count") > 1) & (col("_row_num") > 1), lit("duplicate row")).otherwise(lit(None).cast("string")),
        when((col("fb_discount") < 0) | (col("fb_discount") > 1), lit("fb_discount is less than zero or greater than one")).otherwise(lit(None).cast("string")),
        when(col("_invalid_bookingid").isNotNull(), col("_invalid_bookingid")).otherwise(lit(None).cast("string")),
        when(col("fb_departuredate") > col("fb_arrivaldate"), lit("fb_departuredate is greater than fb_arrivaldate")).otherwise(lit(None).cast("string")),
        when(col("fb_seats") <= 0, lit("fb_seats is zero or negative")).otherwise(lit(None).cast("string"))
    ), lit(""))
)

quarantine_flight_booking_df = standardized_flight_booking_df.filter(
    col("_quarantine_reason").isNotNull()
).drop("_key_count", "_valid_bookingid", "_invalid_bookingid", "_row_num")

clean_flight_booking_df = standardized_flight_booking_df.filter(
    col("fb_bookingid").isNotNull()
    & col("fb_routeid").isNotNull()
    & col("fb_airlineid").isNotNull()
    & col("fb_linenumber").isNotNull()
    & (col("_row_num") == 1)
    & col("_valid_bookingid").isNotNull()
    & ~((col("fb_discount") < 0) | (col("fb_discount") > 1))
    & (col("fb_departuredate") <= col("fb_arrivaldate"))
    & (col("fb_seats") > 0)
).drop("_key_count", "_quarantine_reason", "_valid_bookingid", "_invalid_bookingid", "_row_num")

# Write quarantined records
quarantine_flight_booking_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.flight_booking_quarantine")

# Write clean records
clean_flight_booking_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.flight_booking")

In [0]:
# Reconciliation: bronze row count == silver clean + silver quarantine, per table
from pyspark.sql.functions import *

reconciliation_tables = [
    ("region",),
    ("nation",),
    ("passenger",),
    ("route",),
    ("airline",),
    ("airline_route",),
    ("booking",),
    ("flight_booking",)
]

recon_rows = []
for (tbl,) in reconciliation_tables:
    bronze_count = spark.read.table(f"{catalog}.bronze.{tbl}").count()
    silver_count = spark.read.table(f"{catalog}.silver.{tbl}").count()
    quarantine_count = spark.read.table(f"{catalog}.silver.{tbl}_quarantine").count()
    recon_rows.append((
        tbl,
        bronze_count,
        silver_count,
        quarantine_count,
        silver_count + quarantine_count,
        bronze_count == silver_count + quarantine_count
    ))

recon_df = spark.createDataFrame(
    recon_rows,
    schema="source_table string, bronze_count long, silver_count long, quarantine_count long, silver_plus_quarantine long, balanced boolean"
)

recon_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.reconciliation_summary")